# Bölüm 13: Modelinizi İnce Ayar İle Eğitme

> "Bir tekmeyi 10,000 kez tekrar eden adam değil, 10,000 farklı tekmeyi bir kez tekrar eden adamdan korkarım."
> — **Bruce Lee**, Dövüş Sanatçısı

---

## Öğrenecekleriniz

- İnce ayar ile daha iyi promptlar arasında ne zaman tercih yapılır
- Talimat/yanıt eğitim verisi nasıl hazırlanır
- Kayıp maskeleme neden eğitimi önemli olana odaklar
- LoRA adaptörleri nasıl 48× parametre verimliliği sağlar
- Model davranışı öncesi/sonrası nasıl değerlendirilir

---

## Kurulum

İlk olarak, gerekli paketleri kuralım ve GPU kullanılabilirliğini kontrol edelim.

In [ ]:
# Ek A’da test edilen sürümleri yükle
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== İÇE AKTARMALAR =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# GPU kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU bulunamadı. Eğitim yavaş olacak.")
    print("Runtime > Change runtime type > GPU'ya gidin")

In [ ]:
# ===== YENİDEN ÜRETİLEBİLİRLİK =====
import random
import numpy as np

def set_seed(seed=42):
    """Kullandığımız her rastgele kaynağı tohumla (Bölüm 12'deki yardımcının aynısı)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)


## 1. Bölüm 10-12'den Model Bileşenleri

İlk olarak, önceki bölümlerde oluşturduğumuz MiniGPT modelini getireceğiz.

In [ ]:
# ===== ÇOK BAŞLI DİKKAT (Bölüm 10'dan) =====
class MultiHeadAttention(nn.Module):
    """
    Verimli çok başlı dikkat (tüm başları birlikte yığınlar).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Birleşik QKV projeksiyonu (ayrı ayrı yapmaktan 3 kat verimli!)
        # Neden 3 * d_model? Çünkü Q, K, V'ye aynı anda projeksiyon yapıyoruz
        # bias=True GPT-2 ile eşleşir; böylece önceden eğitilmiş bias'larını sonra yükleyebiliriz
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Çıktı projeksiyonu
        self.out_proj = nn.Linear(d_model, d_model)

        # Dikkat ağırlıklarında dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Adım 1: Q, K, V'ye projeksiyon (hepsi tek seferde!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Adım 2: Q, K, V'ye böl ve çok başlı için yeniden şekillendir =====
        # (batch, seq, 3, num_heads, d_head) şekline yeniden şekillendir
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # (3, batch, num_heads, seq, d_head) düzenine geçir
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Q, K, V'ye böl: her biri (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Adım 3: Ölçeklendirilmiş nokta çarpım dikkati (başlar üzerinde yığınlanmış) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Verildiyse nedensel maskeyi uygula
        if mask is not None:
            # Maskeyi başlar için genişlet: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Değerlerin ağırlıklı toplamı
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Adım 4: Başları birleştir =====
        # (batch, seq, num_heads, d_head) düzenine transpoz et
        attn_output = attn_output.transpose(1, 2)

        # (batch, seq, d_model) şekline yeniden şekillendir; bu başları birleştirir
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Adım 5: Son projeksiyon =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Test et

print("MultiHeadAttention tanımlandı!")

In [ ]:
# ===== İLERİ BESLEMELİ AĞ (Bölüm 10'dan) =====
class FeedForward(nn.Module):
    """
    Konuma göre ileri beslemeli ağ.
    Her konuma bağımsız olarak uygulanır (tüm konumlar için aynı ağırlıklar).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Argümanlar:
            d_model: Model boyutu (GPT-2 small için 768)
            d_ff: İleri beslemeli boyutu (tipik olarak 4 * d_model = 3072)
            dropout: Dropout olasılığı
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), genişlet
        x = F.gelu(x, approximate="tanh")  # Doğrusal olmayan (GPT-2'nin GELU varyantı)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), geri projeksiyon
        return x


# Test et

print("FeedForward tanımlandı!")

In [ ]:
# ===== TRANSFORMER BLOĞU (Bölüm 10'dan) =====
class TransformerBlock(nn.Module):
    """
    Çok başlı dikkat, ileri beslemeli, artık bağlantılar ve
    katman normalizasyonu ile tam Transformer bloğu (GPT-2 gibi ön-norm stili).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Argümanlar:
            d_model: Model boyutu (768)
            num_heads: Dikkat başı sayısı (12)
            d_ff: İleri beslemeli boyutu (3072 = 4 * d_model)
            dropout: Dropout olasılığı
        """
        super().__init__()

        # Katman normalizasyonu (her alt katmandan önce)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Çok başlı dikkat
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # İleri beslemeli ağ
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (her alt katmandan sonra uygulanır)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Argümanlar:
            x: Girdi gömmeleri (batch, seq, d_model)
            mask: Nedensel maske (seq, seq) veya (1, 1, seq, seq)

        Döndürür:
            x: Çıktı gömmeleri (batch, seq, d_model)
            attn_weights: Dikkat ağırlıkları (batch, heads, seq, seq)
        """
        # ===== Artık Bağlantılı Çok Başlı Dikkat =====
        # Ön-norm: Dikkatten ÖNCE normalleştir
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Artık bağlantı

        # ===== Artık Bağlantılı İleri Besleme =====
        # Ön-norm: FFN'den ÖNCE normalleştir
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Artık bağlantı

        return x, attn_weights


# Tam bir Transformer bloğunu test et

print("TransformerBlock tanımlandı!")

In [ ]:
# ===== GPT YAPILANDIRMASI (Bölüm 11'den) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """MiniGPT modeli için yapılandırma."""
    vocab_size: int = 50257      # GPT-2 kelime dağarcığı boyutu
    max_seq_len: int = 1024      # Maksimum bağlam uzunluğu
    embed_dim: int = 768         # Gömme boyutu (d_model)
    num_heads: int = 12          # Dikkat başı sayısı
    num_layers: int = 12         # Transformer bloğu sayısı
    d_ff: int = 3072             # İleri beslemeli gizli boyut (4 * embed_dim)
    dropout: float = 0.1         # Dropout olasılığı

    def __post_init__(self):
        """Başlatmadan sonra yapılandırmayı doğrula."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}), num_heads " \
            f"({self.num_heads}) ile bölünebilir olmalıdır"


In [ ]:
# ===== MİNİGPT MODELİ (Bölüm 11'den) =====
class MiniGPT(nn.Module):
    """
    Minimal bir GPT tarzı dil modeli.
    Gömmeleri, Transformer bloklarını ve dil modelleme başını birleştirir.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token ve Konum Gömmeleri (Bölüm 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer Blokları (Bölüm 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Son LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Dil Modelleme Başı =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Ağırlık Bağlama =====
        # Gömme ve çıktı projeksiyonu arasında ağırlıkları paylaş
        self.lm_head.weight = self.token_embed.weight

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Ağırlıkları küçük rastgele değerlerle başlat."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Model üzerinden ileri geçiş.

        Argümanlar:
            token_ids: Girdi token ID'leri (batch, seq)
            return_attention: Dikkat ağırlıklarının döndürülüp döndürülmeyeceği

        Döndürür:
            logits: Kelime dağarcığı skorları (batch, seq, vocab_size)
            attention_weights: Blok başına dikkat ağırlıklarının isteğe bağlı listesi
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Gömmeler =====
        # Token gömmeleri
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Konum gömmeleri
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Birleştir ve dropout uygula
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Nedensel Maske Oluştur =====
        # Alt üçgen: her konum yalnızca önceki konumlara dikkat edebilir
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Transformer Bloklarından Geçir =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Son LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Dil Modelleme Başı =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        input_ids'i devam ettiren token'lar üret (istem çıktıya dahildir).

        Argümanlar:
            input_ids: İstem token ID'leri (batch, seq)
            max_new_tokens: Kaç yeni token üretilecek
            temperature: Örnekleme sıcaklığı (yalnızca do_sample=True iken kullanılır)
            do_sample: Dağılımdan örnekle; False = açgözlü argmax
            eos_token_id: Her dizi bu token'ı üretirse erken dur

        Döndürür:
            Token ID'leri (batch, seq + yeni token'lar)
        """
        self.eval()

        # Konum gömmeleri yalnızca max_seq_len konumu kapsar
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} token) + max_new_tokens "
                f"({max_new_tokens}) modelin {limit} token'lık bağlamını "
                "aşıyor. Prompt'u kısaltın ya da max_new_tokens'ı düşürün.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


## 2. Bölüm 12 Temel Modelini Alma (kontrol noktası içe aktarma)

Kitap **Bölüm 12 Shakespeare kontrol noktasını** ince ayarlar; bu notebook da öyle. Temel değerlendirme, LoRA eğitimi ve adaptör yeniden yükleme **aynı temel ağırlıkları** kullanır. İki dürüst yol:

- **Yol A (tercih edilen):** Bölüm 12’de kendi eğittiğiniz `best_model.pt` dosyasını bu defterin çalışma dizinine kopyalayın. Colab kullanıyorsanız dosyayı önce Bölüm 12’nin Dosyalar panelinden indirin, sonra bu defterin Dosyalar panelinden yükleyin.
- **Yol B (dosyasız yeni Colab):** aşağıdaki hücre, Bölüm 12 tarifini TinyShakespeare üzerinde **1 dönemlik mini eğitim** olarak çalıştırır ve `best_model.pt` olarak kaydeder. Bu temel, kitabın 3 dönemlik çalıştırmasından zayıftır; kayıplar farklı olur ama her hücre bu tek tutarlı temeli kullanır.

Rastgele başlatılmış temel, notebook'un sonunda *ayrı ve etiketli* bir deneydir; varsayılan asla değildir.

In [ ]:
# ===== BÖLÜM 12 KONTROL NOKTASI İÇE AKTARMA =====
# Aşağıdaki her hücre (temel, LoRA eğitimi, yeniden yükleme) BU temeli kullanır.
import os
import urllib.request

config = GPTConfig(
    vocab_size=50257, max_seq_len=128, embed_dim=256,
    num_heads=4, num_layers=4, d_ff=1024, dropout=0.1
)

# Load tokenizer. use_fast=True gives us GPT2TokenizerFast, whose
# character offsets we need for loss masking below.
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Yol A: Colab’da kendi kontrol noktanızı yüklemek için aşağıdaki iki satırın yorum işaretlerini kaldırın.
# from google.colab import files
# files.upload()  # Kendi best_model.pt dosyanızı seçin.

if not os.path.exists("best_model.pt"):
    # Route B: no checkpoint on hand -> Chapter 12 mini-training (1 epoch).
    # Same recipe as the Chapter 12 notebook: 512-char chunks, batch 32,
    # AdamW + linear warmup, pads masked via the attention mask.
    print("No best_model.pt found: running Chapter 12 mini-training (1 epoch)...")
    from transformers import get_linear_schedule_with_warmup

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "shakespeare.txt")
    text = open("shakespeare.txt").read()

    def create_lm_labels(token_ids):
        inputs = token_ids[:, :-1]
        targets = token_ids[:, 1:].clone()
        return inputs, targets  # pads masked in the loop via attention_mask

    class TextDataset(Dataset):
        def __init__(self, text, chunk_size=512):
            self.chunks = [text[i:i + chunk_size]
                           for i in range(0, len(text) - chunk_size + 1, chunk_size)]
        def __len__(self):
            return len(self.chunks)
        def __getitem__(self, idx):
            return self.chunks[idx]

    def ch12_collate(batch, tokenizer, max_length=128):
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    ds = TextDataset(text, chunk_size=512)
    n_train = int(0.9 * len(ds))
    train_ds, val_ds = torch.utils.data.random_split(
        ds, [n_train, len(ds) - n_train],
        generator=torch.Generator().manual_seed(42))
    collate12 = partial(ch12_collate, tokenizer=tokenizer, max_length=128)
    train_loader12 = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate12)

    set_seed(42)
    model = MiniGPT(config).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-4)
    # Tek dönemlik koşuda ısınma tüm eğitimi kaplamasın.
    total_steps12 = len(train_loader12)
    warmup_steps12 = min(100, max(1, total_steps12 // 10))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps12,
        num_training_steps=total_steps12)

    model.train()
    total_loss = 0
    for input_ids, attention_mask in tqdm(train_loader12, desc="Ch12 mini-training"):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # ignore pads in the loss
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=-100)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Mini-training done: avg loss {total_loss / len(train_loader12):.4f}")
    # Weights-only artifact: no optimizer/RNG state, safe to reload anywhere
    torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")
    print("Saved best_model.pt (1-epoch mini base; weaker than the book's 3-epoch run)")

# Load the checkpoint (either route) into THE base model used everywhere below.
# Chapter 12's resume checkpoint also stores RNG states (NumPy objects the
# safe loader rejects), so allow exactly those classes, nothing else.
import numpy as np
torch.serialization.add_safe_globals([np._core.multiarray._reconstruct,
                                      np.ndarray, np.dtype])
base_model = MiniGPT(config).to(device)
try:
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=True)
except Exception:
    # Older checkpoints may carry further NumPy state types; this file is
    # one YOU trained in Chapter 12, so a full load of it is acceptable.
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()
BASE_SOURCE = "Chapter 12 checkpoint (best_model.pt)"
print(f"Base model ready: {sum(p.numel() for p in base_model.parameters()):,} parameters")
print(f"Base weights: {BASE_SOURCE}")

## 3. İnce Ayar Veri Kümesi Hazırlama

TechStartup Inc FAQ için talimat/yanıt çiftleri oluşturalım.

In [ ]:
# ===== SSS VERİ KÜMESİ (kitapla aynı: paraphrase GRUBU bölünmesi) =====
import json
import random

import json
import random

# Seven hand-written FAQ pairs for our fictional company
base_examples = [
    {"instruction": "What does TechStartup Inc do?",
     "response": "TechStartup Inc builds AI-powered tools."},

    {"instruction": "When was TechStartup Inc founded?",
     "response": "TechStartup Inc was founded in 2020."},

    {"instruction": "Where is TechStartup Inc located?",
     "response": "TechStartup Inc is in Amsterdam."},

    {"instruction": "What is SmartScheduler?",
     "response": "SmartScheduler is our AI calendar assistant."},

    {"instruction": "How much does SmartScheduler cost?",
     "response": "10 euros/month individual, 8 euros/user teams."},

    {"instruction": "How do I reset my password?",
     "response": "Click 'Forgot Password' on the login page."},

    {"instruction": "Can I export my data?",
     "response": "Yes, go to Settings > Data > Export."},
]


# Rephrasings per question (same order as base_examples)
paraphrases = [
    ["What does TechStartup Inc do?", "What is TechStartup Inc's business?",
     "Tell me what TechStartup Inc makes.", "What kind of company is TechStartup Inc?"],
    ["When was TechStartup Inc founded?", "What year was TechStartup Inc started?",
     "How old is TechStartup Inc?", "When did TechStartup Inc begin?"],
    ["Where is TechStartup Inc located?", "Where is TechStartup Inc based?",
     "Which city is TechStartup Inc in?", "Where are TechStartup Inc's offices?"],
    ["What is SmartScheduler?", "Tell me about SmartScheduler.",
     "What does SmartScheduler do?", "Describe SmartScheduler."],
    ["How much does SmartScheduler cost?", "What is SmartScheduler's price?",
     "What does SmartScheduler cost per month?", "How much is a SmartScheduler subscription?"],
    ["How do I reset my password?", "I forgot my password, what do I do?",
     "How can I change my password?", "What is the password reset process?"],
    ["Can I export my data?", "How do I export my data?",
     "Is data export possible?", "Where is the data export option?"],
]

def expand_faq(base_examples, paraphrases, seed=42):
    """Deterministically expand the FAQ pairs, splitting by paraphrase GROUP.

    Each paraphrase plus its four polite-prefix variants forms one group
    (7 questions x 4 paraphrases = 28 groups of 4). We shuffle the GROUPS
    and hold out 5 whole groups (20 examples) BEFORE any train/eval string
    exists in the other split, so no eval phrasing, prefixed or not, was
    ever trained on. Train takes 80 of the remaining 92 examples.
    """
    prefixes = ["", "Please tell me: ", "Quick question: ", "Could you tell me: "]
    groups = []
    for qid, (example, variants) in enumerate(zip(base_examples, paraphrases)):
        for paraphrase in variants:
            groups.append((qid, paraphrase,
                           [{"instruction": p + paraphrase,
                             "response": example["response"]} for p in prefixes]))
    rng = random.Random(seed)
    rng.shuffle(groups)

    # Hold out 5 whole groups, but never the LAST group of a question:
    # every intent must keep at least one phrasing in training.
    left = {qid: sum(1 for q, _, _ in groups if q == qid)
            for qid in range(len(base_examples))}
    eval_groups, train_groups = [], []
    for qid, paraphrase, examples in groups:
        if len(eval_groups) < 5 and left[qid] > 1:
            eval_groups.append((qid, paraphrase, examples))
        else:
            train_groups.append((qid, paraphrase, examples))
        left[qid] -= 1

    eval_data = [ex for _, _, g in eval_groups for ex in g]      # 5 x 4 = 20
    train_pool = [ex for _, _, g in train_groups for ex in g]    # 23 x 4 = 92
    rng.shuffle(train_pool)
    train_data = train_pool[:80]
    return train_data, eval_data

train_data, eval_data = expand_faq(base_examples, paraphrases)

# Assert 1: exact question strings are disjoint (the classic leakage check)
train_questions = {ex["instruction"] for ex in train_data}
assert all(ex["instruction"] not in train_questions for ex in eval_data)

# Assert 2: paraphrase GROUPS are disjoint. Strip the polite prefixes and
# check that no eval phrasing appears in training in ANY prefixed form.
def core_question(instruction):
    for p in ["Please tell me: ", "Quick question: ", "Could you tell me: "]:
        if instruction.startswith(p):
            return instruction[len(p):]
    return instruction

train_cores = {core_question(q) for q in train_questions}
assert all(core_question(ex["instruction"]) not in train_cores for ex in eval_data)

# Coverage, honestly: every intent keeps training phrasings, but all seven
# ANSWERS still appear on both sides. This tests unseen phrasings, not unseen intents.
train_intents = {ex["response"] for ex in train_data}
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"Intents covered in training: {len(train_intents)}/7")
print(f"Eval phrasings seen in training (prefixes stripped): "
      f"{sum(core_question(ex['instruction']) in train_cores for ex in eval_data)}/20")
# Output: Train: 80, Eval: 20
# Output: Intents covered in training: 7/7
# Output: Eval phrasings seen in training (prefixes stripped): 0/20


def save_as_jsonl(examples, filepath):
    """Save examples in JSONL format (one JSON object per line)."""
    with open(filepath, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

def load_jsonl(filepath):
    """Load examples from JSONL format."""
    examples = []
    with open(filepath, 'r') as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

# Bu hücre faq_dataset.jsonl dosyasını çalışma dizinine yazar. Paketteki İngilizce örnek: data/faq_dataset.jsonl.
save_as_jsonl(train_data + eval_data, 'faq_dataset.jsonl')


In [ ]:
# ===== BİÇİMLENDİRME + YIĞINLAMA (kitaptan) =====
# Özel token'ları tanımla
# Define special tokens
INST_START = "[INST]"
INST_END = "[/INST]"

def build_prompt(instruction):
    """The ONE prompt format shared by training and generation.

    Ends right after [/INST], with no trailing space: a trailing space
    can tokenize differently alone than glued to the answer's first
    word (" Tech"), which would make inference feed the model a prefix
    it never saw in training.
    """
    return f"{INST_START} {instruction} {INST_END}"

def format_example(instruction, response, tokenizer):
    """
    Format an instruction/response pair for training.

    Needs a fast tokenizer (GPT2TokenizerFast, which our setup loads):
    only fast tokenizers report character offsets for each token.

    Returns:
        input_ids: Full tokenized sequence
        labels: Same as input_ids, but with -100 for instruction tokens
    """
    # Format: [INST] instruction [/INST] response + end-of-text.
    # The EOS token teaches the model where answers END, so generation
    # can stop there instead of rambling on after the answer.
    prompt = build_prompt(instruction)
    full_text = prompt + " " + response + tokenizer.eos_token

    # Tokenize, asking for each token's character span in full_text
    enc = tokenizer(full_text, return_offsets_mapping=True)
    input_ids = torch.tensor(enc["input_ids"])

    # Find where the response starts: the first token whose character
    # span lies fully past the prompt. BPE glues the space we added
    # onto the answer's first word (" Tech"), so that token belongs to
    # the answer; counting the prompt's tokens alone would mask it.
    response_start = next(i for i, (_, end) in enumerate(enc["offset_mapping"])
                          if end > len(prompt))

    # Create labels: -100 for instruction tokens (ignored by loss)
    labels = input_ids.clone()
    labels[:response_start] = -100  # Mask instruction tokens

    return input_ids, labels


import torch
from functools import partial
from torch.utils.data import DataLoader

PAD_ID = tokenizer.eos_token_id  # GPT-2 metin sonu ile doldurur

def collate_fn(examples, tokenizer, max_length=128):
    """Talimat/yanıt çiftlerini doldurulmuş input_ids + labels'a tokenize et."""
    batch_ids, batch_labels = [], []
    for ex in examples:
        ids, labels = format_example(ex['instruction'], ex['response'], tokenizer)
        ids, labels = ids[:max_length], labels[:max_length]
        pad = max_length - len(ids)
        ids = torch.cat([ids, torch.full((pad,), PAD_ID, dtype=torch.long)])
        labels = torch.cat([labels, torch.full((pad,), -100, dtype=torch.long)])
        batch_ids.append(ids)
        batch_labels.append(labels)
    return {'input_ids': torch.stack(batch_ids),
            'labels': torch.stack(batch_labels)}

collate = partial(collate_fn, tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate)


# Bir yığını kontrol et
batch = next(iter(train_loader))
print(f"Input IDs şekli: {batch['input_ids'].shape}")
print(f"Etiketler şekli: {batch['labels'].shape}")
print(f"İlk örnekte maskelenmiş konumlar: {(batch['labels'][0] == -100).sum().item()}")

## 4. Temel Değerlendirme (İnce Ayar ÖNCESİ)

Temel modelin FAQ promptlarına nasıl yanıt verdiğini görelim.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Bir talimat promptuna yanıt üret."""
    model.eval()
    full_prompt = build_prompt(prompt)
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# FAQ promptlarında test et
test_prompts = [
    "What does TechStartup Inc do?",
    "How do I reset my password?",
    "What is SmartScheduler?"
]

print(f"İNCE AYAR ÖNCESİ ({BASE_SOURCE}):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nS: {prompt}")
    print(f"C: {response[:100]}...")
print("\n(Shakespeare tadında saçmalık - Bölüm 12 temeli TechStartup Inc")
print("hakkında hiçbir şey bilmiyor ve 16M sıfırdan model akıcı")
print("İngilizceyi hiç öğrenmedi. Dürüst beklentiler!)")


## 5. LoRA Bileşenlerini Tanımlama

Temel ağırlıkları değiştirmeden eğitilebilir parametreler ekleyen LoRA adaptör katmanını oluşturalım.

### Neden Tam İnce Ayar Yerine LoRA?

Tam ince ayarın bir sorunu var: **felaket unutma**. Yeni bir görev için tüm ağırlıkları güncellediğinizde, model ön eğitim sırasında öğrendiklerini "unutabilir". LoRA, temel ağırlıkları *dondurarak* ve yanlarına küçük eğitilebilir matrisler *ekleyerek* bunu büyük ölçüde azaltır.

Bunu okuma gözlüğü gibi düşünün: gözleriniz (temel model) aynı kalır, ancak gözlük (LoRA) belirli görevler için küçük bir düzeltme ekler.

In [ ]:
# ===== LORA DOĞRUSAL KATMAN (makale kuralı: A aşağı, B yukarı) =====
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    Düşük Ranklı Adaptasyon (LoRA) ile doğrusal katman.

    Bunu mevcut bir katmana küçük bir "düzeltme notu" eklemek gibi düşün.
    Temel katman donuk kalırken iki küçük matris ayarlamaları öğrenir.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        # Orijinal katman (bunu donduracağız); bias GPT-2'nin qkv_proj'i ile eşleşir
        self.base = nn.Linear(in_features, out_features)

        # LoRA matrisleri (makale kuralı: A aşağı, B yukarı)
        # A: (in_features, rank) - "aşağı" projeksiyon
        # B: (rank, out_features) - "yukarı" projeksiyon
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        # A'yı küçük rastgele değerlerle (Kaiming), B'yi sıfırlarla başlat.
        # Transpoze görünüm üzerinden başlat ki Kaiming (rank, in_features) görsün:
        # nn.Linear(in_features, rank) ağırlık şekli bu; fan_in böylece
        # in_features olur. Görünüm üzerinde yerinde yazmak lora_A'yı günceller.
        nn.init.kaiming_uniform_(self.lora_A.T, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        # Ölçekleme faktörü
        self.scale = alpha / rank

    def freeze_base(self):
        """Temel katmanı dondur; böylece yalnızca LoRA matrisleri eğitilir."""
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

    def forward(self, x):
        # x şekli: (batch, sequence, in_features)

        # Donuk ağırlıklardan orijinal çıktı
        base_output = self.base(x)  # (batch, seq, out_features)

        # LoRA yolu: x @ A @ B, ölçeklenmiş
        # x @ A: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  [sıkıştırıldı!]
        # ... @ B: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  [geri genişletildi]
        lora_output = (x @ self.lora_A @ self.lora_B) * self.scale

        return base_output + lora_output

print("LoRALinear tanımlandı!")

# Parametre tasarrufunu göster
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nParametre karşılaştırması (256x256 katman):")
print(f"  Tam ince ayar: {full_params:,} parametre")
print(f"  LoRA (rank=8):    {lora_params:,} parametre")
print(f"  Tasarruf:          {full_params/lora_params:.1f}x daha az!")

## 6. Modele LoRA Uygulama

Dikkat projeksiyonlarını LoRA versiyonlarıyla değiştir ve temel modeli dondur.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    Dikkat projeksiyonlarına LoRA ekle.

    Birleşik sorgu/anahtar/değer projeksiyonu olan qkv_proj'i hedefliyoruz.
    Adaptör, Q, K ve V için ortak düzeltmeler öğrenir.
    """
    # Adım 1: Önce TÜM parametreleri dondur
    for param in model.parameters():
        param.requires_grad = False

    # Adım 2: Her transformer bloğunda qkv_proj'i değiştir
    # Bölüm 11'de oluşturduğumuz bloklar üzerinde açıkça geziyoruz
    for block in model.blocks:
        # Dikkat katmanının QKV projeksiyonunu al
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # LoRA güçlendirilmiş yedeği modelle aynı cihazda oluştur
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)
        lora_qkv = lora_qkv.to(original_qkv.weight.device)

        # Önceden eğitilmiş ağırlıkları (ve bias'ı!) temel katmana kopyala
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()
        lora_qkv.base.bias.data = original_qkv.bias.data.clone()

        # Temeli dondur (LoRA matrisleri eğitilebilir kalır)
        lora_qkv.freeze_base()
        if lora_qkv.base.bias is not None:
            lora_qkv.base.bias.requires_grad = False

        # Katmanı değiştir
        block.attn.qkv_proj = lora_qkv

    return model


# Kitapla aynı temel: yukarıda içe aktarılan Bölüm 12 kontrol noktası.
# Temel değerlendirme, eğitim ve yeniden yükleme bu ağırlıkları paylaşır;
# ÖNCE/SONRA karşılaştırmaları dürüst kalır. (Yol B mini eğitim yaptıysa
# kayıplar kitabın 3 dönemlik kontrol noktası sayılarından farklı olur.)
set_seed(42)  # adaptör başlatmadan ve yığın karıştırmadan ÖNCE (eğitimden sonra tohumlamak hiçbir şeyi yeniden üretmez)
model = MiniGPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model = add_lora_to_model(model, rank=8, alpha=16)

# Parametreleri say
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Toplam parametreler:     {total:,}")
print(f"Eğitilebilir (LoRA):     {trainable:,}")
print(f"Eğitilebilir yüzdesi: {100*trainable/total:.2f}%")

## 7. İnce Ayar Döngüsü

Maskeli kayıp ile 5 adımlı tarif kullanarak eğit.

In [ ]:
import torch.nn.functional as F

def compute_masked_loss(logits, labels):
    """
    Etiketlerin -100 olduğu konumları göz ardı ederek çapraz entropi kaybını hesapla.
    
    Bu, bir sınavın sadece cevap kısmını değerlendirmek gibidir,
    prompttan kopyalanan soruyu değil.
    """
    # Sonraki token tahmini için kaydır (Bölüm 12 ile aynı)
    # .contiguous() belleğin .view() için sıralı olarak düzenlendiğinden emin olur
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # ignore_index=-100 ile çapraz entropi
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # PyTorch bu konumları göz ardı eder
    )
    
    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    Maskeli kayıpla 5 adımlı tarifi kullanarak bir dönem ince ayar yap.
    """
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # ===== 5 ADIMLI TARİF =====
        
        # Adım 1: Gradyanları sıfırla
        optimizer.zero_grad()
        
        # Adım 2: İleri geçiş
        logits = model(input_ids)
        
        # Adım 3: MASKELİ kayıp hesapla
        loss = compute_masked_loss(logits, labels)
        
        # Adım 4: Geri geçiş
        loss.backward()
        
        # Adım 5: Ağırlıkları güncelle
        optimizer.step()
        
        total_loss += loss.item()

    return total_loss / len(dataloader)

print("Eğitim fonksiyonları tanımlandı!")

In [ ]:
# ===== EĞİTİM =====
num_epochs = 3
learning_rate = 1e-3  # LoRA adaptörleri tam ince ayardan daha yüksek LR'ye toleranslıdır

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate  # same optimizer as the book: plain AdamW, no weight decay
)

train_losses = []
print(f"{len(train_data)} örnek üzerinde {num_epochs} dönem için eğitiliyor\n")
for epoch in range(num_epochs):
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    print(f"Dönem {epoch+1}: maskeli kayıp {loss:.4f}")

print("\nİnce ayar tamamlandı!")
print("Not: sıfırdan eğitilmiş 16M temel modelle anlamsız üretimler bekleyin;")
print("kazanç, kayıp düşüşü ve adaptör mekaniğidir. Akıcı yanıtlar için")
print("önceden eğitilmiş GPT-2 ağırlıklarından başlayın (Bölüm 11'in yükleyicisi).")

## 8. Eğitim İlerlemesini Görselleştir

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Dönem', fontsize=12)
plt.ylabel('Eğitim Kaybı', fontsize=12)
plt.title('İnce Ayar Kayıp Eğrisi', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nKayıp {train_losses[0]:.4f}'den {train_losses[-1]:.4f}'e düştü")
print(f"İyileştirme: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Değerlendirme (İnce Ayar SONRASI)

Dürüst kısım: neyin değiştiğini ölçmek. Sıfırdan eğitilmiş 16M temel modelimizle üretimler ÖNCE de SONRA da anlamsız kalır (akıcı İngilizceyi hiç öğrenmedi). Ölçülebilir kazanç, eğitim kaybındaki düşüş ve adaptör mekaniğidir. Akıcı bir önce/sonra karşılaştırmasını GPT-2 gibi önceden eğitilmiş bir temel (Bölüm 11'in yükleyicisi) üretir.

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """
    Evaluate fine-tuned model on test examples.
    """
    model.eval()
    results = []

    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']

        # Generate from the SAME prefix format training used
        prompt = build_prompt(instruction)
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

        # Held-out loss: teacher-force the CORRECT answer through the
        # model and grade only response tokens (same masked loss as
        # training, but on data the adapters never saw)
        full_ids, full_labels = format_example(instruction, expected, tokenizer)
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(device))
            resp_loss = compute_masked_loss(logits, full_labels.unsqueeze(0).to(device))

        output = model.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

        # Decode only the new tokens, dropping the EOS terminator itself
        new_tokens = output[0][input_ids.shape[1]:].tolist()
        if tokenizer.eos_token_id in new_tokens:
            new_tokens = new_tokens[:new_tokens.index(tokenizer.eos_token_id)]
        generated = tokenizer.decode(new_tokens).strip()

        # Exact match
        is_exact = generated.lower() == expected.lower()

        # Word overlap
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)

        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap,
            'response_loss': resp_loss.item()
        })

    # Summary
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)

    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'avg_response_loss': sum(r['response_loss'] for r in results) / len(results),
        'detailed_results': results
    }

# Ayrılan bölünme üzerinde değerlendir
print("Test kümesi üzerinde değerlendiriliyor...")
results = evaluate_model(model, eval_data, tokenizer, device)

print(f"\n===== DEĞERLENDİRME SONUÇLARI =====")
print(f"Tam Eşleşme Doğruluğu: {results['exact_match_accuracy']*100:.1f}%")
print(f"Ortalama Kelime Örtüşmesi: {results['average_word_overlap']*100:.1f}%")
print("(Sıfırdan eğitilmiş temel modelle %0'a yakın beklenir; yukarıdaki nota bakın.)")
print(f"Ayrılan kümede yanıt kaybı: {results['avg_response_loss']:.2f}")


In [ ]:
# Detaylı sonuçları göster
print("\n===== DETAYLI TEST SONUÇLARI =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} S: {r['instruction']}")
    print(f"  Beklenen:  {r['expected']}")
    print(f"  Üretilen: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Üretilen: {r['generated']}")
    print(f"  Örtüşme:   {r['word_overlap']*100:.0f}%")

In [ ]:
def show_before_after(base_model, finetuned_model, test_prompts, tokenizer, device):
    """
    Side-by-side comparison: base model vs fine-tuned model.
    """
    print("=" * 70)
    print("BEFORE vs AFTER Fine-Tuning")
    print("=" * 70)

    for prompt in test_prompts:
        full_prompt = build_prompt(prompt)
        input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)

        base_output = base_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)
        ft_output = finetuned_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)

        base_text = tokenizer.decode(base_output[0][len(input_ids[0]):])
        ft_text = tokenizer.decode(ft_output[0][len(input_ids[0]):])

        print(f"\nPrompt: {prompt}")
        print(f"  BEFORE: {base_text.strip()}")
        print(f"  AFTER:  {ft_text.strip()}")
        print("-" * 70)

# ÖNCE modeli: temel ağırlıkların birebir kopyası (uyarlanmamış)
base_model_fresh = MiniGPT(config).to(device)
base_model_fresh.load_state_dict(base_model.state_dict())

test_prompts = ["TechStartup Inc ne yapıyor?", "Şifremi nasıl sıfırlarım?"]
set_seed(42)  # aşağıdaki örneklemeli karşılaştırma yeniden üretilebilir olsun diye tekrar tohumla
show_before_after(base_model_fresh, model, test_prompts, tokenizer, device)

print("\nİki taraf da anlamsız: 16M sıfırdan model akıcı İngilizceyi hiç öğrenmedi.")
print("Ölçülebilir kazanç, kayıp düşüşü + çalışan adaptör mekaniğidir.")
print("Akıcı bir önce/sonra görmek için önce GPT-2 ağırlıklarını yükleyin (Bölüm 11).")

## 10. LoRA Ağırlıklarını Kaydet ve Yükle

In [ ]:
def save_lora_weights(model, filepath):
    """
    Yalnızca LoRA parametrelerini kaydet (minik dosya!).
    """
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)

    # Dosya boyutunu göster
    import os
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"LoRA ağırlıkları kaydedildi: {filepath} ({size_mb:.2f} MB)")



# LoRA ağırlıklarını kaydet
save_lora_weights(model, 'techstartup_lora.pt')

# Tam model boyutuyla karşılaştır
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
lora_size = os.path.getsize('techstartup_lora.pt') / (1024 * 1024)
print(f"Tam model kaydedildi: full_model.pt ({full_size:.1f} MB)")
print(f"\nLoRA tam modelden {full_size/lora_size:.0f}x daha küçük!")

In [ ]:
def load_lora_weights(model, filepath):
    """
    LoRA katmanları zaten olan bir modele LoRA ağırlıklarını yükle.
    """
    lora_state_dict = torch.load(filepath, weights_only=True)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"LoRA ağırlıkları yüklendi: {filepath}")



# Yüklemeyi göster
print("\n===== Kaydetme/Yükleme Testi =====")
# Adaptör yalnızca üzerinde eğitildiği temel ile anlamlıdır. Bu notebook
# temel kontrol noktası kaydetmediği için ÖNCE bellekte hâlâ duran temel
# Adaptör yalnızca üzerinde eğitildiği tam temel ile anlamlıdır:
# ÖNCE best_model.pt'yi yeniden yükleyin, sonra LoRA yapısını, sonra adaptörü.
new_model = MiniGPT(config).to(device)
new_model.load_state_dict(checkpoint["model_state_dict"])  # önce temel!
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

before_load = generate_response(new_model, "TechStartup Inc ne yapıyor?", tokenizer)
print(f"LoRA yüklemeden önce: {before_load[:60]}...")

load_lora_weights(new_model, 'techstartup_lora.pt')

after_load = generate_response(new_model, "TechStartup Inc ne yapıyor?", tokenizer)
print(f"LoRA yükledikten sonra:  {after_load[:60]}...")
# Sağlama: yeniden yüklenen model, eğitilmiş modelle aynı logitleri üretmeli
probe = tokenizer.encode("[INST] SmartScheduler nedir? [/INST]",
                         return_tensors="pt").to(device)
model.eval(); new_model.eval()
with torch.no_grad():
    torch.testing.assert_close(new_model(probe), model(probe), atol=1e-5, rtol=0)
print("Yeniden yükleme eşitlik kontrolü geçti")
print("\nLoRA adaptörleri başarıyla kaydedildi ve yüklendi!")


# ===== BİRLEŞTİRME: LoRAyı temele katla ve sarmalayıcıları KALDIR =====
def merge_lora_into_base(model):
    """
    LoRAyı temel ağırlıklara katla ve adaptör sarmalayıcılarını KALDIR;
    böylece birleşik model yeniden düz bir MiniGPT olur (kalıcı).
    """
    for block in model.blocks:
        lora = block.attn.qkv_proj
        if isinstance(lora, LoRALinear):
            # Düzeltmeyi temel ağırlığa katla: W + (A @ B * scale).T
            delta = lora.lora_A @ lora.lora_B * lora.scale
            lora.base.weight.data += delta.T  # Linear kuralı için transpoze
            # Sarmalayıcıyı düz birleşik katmanla değiştir
            block.attn.qkv_proj = lora.base

    print("LoRA ağırlıkları temel modele birleştirildi")

# Birleştir ve doğrula: aynı logitler, adaptör parametreleri gitti.
# NOT: birleştirme kalıcıdır; bu hücreyi yalnızca yukarıdaki karşılaştırmalardan sonra çalıştırın.
probe = tokenizer.encode("[INST] SmartScheduler nedir? [/INST]",
                         return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    adapter_logits = model(probe)

merge_lora_into_base(model)

with torch.no_grad():
    merged_logits = model(probe)
torch.testing.assert_close(merged_logits, adapter_logits, atol=1e-5, rtol=0)

total = sum(p.numel() for p in model.parameters())
print(f"Birleşik model parametreleri: {total:,} (adaptörler kaldırıldı)")
assert not any(isinstance(m, LoRALinear) for m in model.modules())
print("Birleştirme eşitlik kontrolü geçti")
# Neden transpoze? forward x @ A @ B hesaplar; güncelleme (in, out) şeklindedir;
# nn.Linear ağırlıkları (out, in) saklar, bu yüzden delta.T. Sarmalayıcıyı değiştirmek
# (adaptörleri sıfırlamak değil) iki ölü adaptör matris çarpımını kaldırır.


## Özet

**Oluşturduklarımız:**

1. **Kayıp maskeleme ile talimat/yanıt veri kümesi**
2. **Parametrelerin %1'inden azını eğiten LoRA adaptörleri**
3. **5 adımlık tarif kullanan ince ayar döngüsü**
4. **Değerlendirme**: önce/sonra davranışını karşılaştırma (dürüstçe: kayıp düşer, sıfırdan eğitilmiş temelde metin anlamsız kalır)
5. **Dağıtım için adaptör kaydetme/yükleme**

**Temel içgörüler:**

- Kayıp maskeleme eğitimi önemli olana odaklar (yanıtlar, talimatlar değil)
- LoRA 48× daha az eğitilebilir parametre ile harika sonuçlar elde eder
- Temel ağırlıkları dondurmak **felaket unutmayı** (önceden eğitilmiş bilgiyi kaybetmeyi) büyük ölçüde azaltır
- Verinin kalitesi ince ayar için miktardan daha önemlidir
- Adaptörler küçüktür ve farklı görevler için değiştirilebilir

**Sırada:** Bölüm 14, prompt mühendisliğini keşfedecek - model ağırlıklarını değiştirmeden daha iyi çıktılar elde etme!

## İsteğe Bağlı Deney: Rastgele Temel Varyantı (açıkça etiketli)

Yukarıdaki ana akış her yerde Bölüm 12 kontrol noktasını kullanır. Bu deney, kontrol noktasının neden önemli olduğunu göstermek için LoRA tarifini **rastgele başlatılmış bir temel** üzerinde tekrarlar. Kitabın çalıştırması DEĞİLDİR: burada temel, eğitim ve yeniden yükleme aynı rastgele ağırlıkları paylaşır ve kontrol noktası akışından ayrı tutulur.

In [ ]:
# ===== RASTGELE TEMEL VARYANTI (etiketli deney, kitabın çalıştırması DEĞİL) =====
set_seed(42)
rand_base = MiniGPT(config).to(device)  # random init, NO checkpoint
rand_model = MiniGPT(config).to(device)
rand_model.load_state_dict(rand_base.state_dict())  # same random weights
rand_model = add_lora_to_model(rand_model, rank=8, alpha=16)

rand_optimizer = AdamW([p for p in rand_model.parameters() if p.requires_grad],
                       lr=1e-3, weight_decay=0.01)
rand_losses = []
for epoch in range(3):
    loss = finetune_epoch(rand_model, train_loader, rand_optimizer, device)
    rand_losses.append(loss)
    print(f"[random base] Epoch {epoch+1}: masked loss {loss:.4f}")

print(f"\nCheckpoint-base losses: {[f'{l:.4f}' for l in train_losses]}")
print(f"Random-base losses:     {[f'{l:.4f}' for l in rand_losses]}")
print("Same recipe, different base: the numbers depend on the starting weights,")
print("which is exactly why the adapter must be reloaded onto its OWN base.")

## Alıştırmalar

### Alıştırma 1: Farklı LoRA Rankları

rank=4 ve rank=16'yı deneyin. Eğitim ve sonuçları nasıl etkiliyor?

In [ ]:
# KODUNUZ BURAYA
# 1. rank=4 ile model oluştur
# 2. 3 dönem için eğit
# 3. Kaybı ve değerlendirmeyi rank=8 ile karşılaştır

### Alıştırma 2: Kendi FAQ'nizi Oluşturun

Bildiğiniz bir konu hakkında veri kümesi oluşturun (okul, hobi, vb).

In [ ]:
# KODUNUZ BURAYA
# 1. 30+ talimat/yanıt çifti oluştur
# 2. Modeli ince ayar yap
# 3. Kendi sorularınızla test et

### Alıştırma 3: Kayıp Maskeleme Ablasyonu

Talimat tokenlerini maskeLEMEzsek ne olur?

In [ ]:
# KODUNUZ BURAYA
# 1. Veri kümesini talimat tokenlerini MASKELEMEmek için değiştir (tüm etiketler = gerçek tokenler)
# 2. Modeli eğit
# 3. Sonuçları maskelenmiş versiyonla karşılaştır
# 4. Ne gözlemliyorsunuz?